#Ticket Verilerinin Hazırlanması

Bu notebook, Mendeley Help Desk Tickets verisindeki yazışmaları ticket bazında birleştirir ve RAG için hazırlanan 10 pilot kaydı inceler. Sorun, kategori, çözüm ve sonuç alanları LLM desteğiyle hazırlanmış; kaynak yorum numaraları ve eksik bilgiler korunmuştur. Chunk oluşturma, embedding ve veritabanına kayıt sonraki aşamalardır.

In [1]:
import csv
from pathlib import Path

veri_yolu = Path("sample_utterances.csv")

with veri_yolu.open(encoding="utf-8-sig", newline="") as dosya:
    okuyucu = csv.DictReader(dosya)
    satirlar = list(okuyucu)
    sutunlar = okuyucu.fieldnames

print("Sütunlar:", sutunlar)
print("Satır sayısı:", len(satirlar))
print("Ticket sayısı:", len({satir["issueid"] for satir in satirlar}))

Sütunlar: ['issueid', 'id', 'comment_seq', 'utr_seq', 'author', 'created', 'is_private', 'actionbody', 'author_role']
Satır sayısı: 30104
Ticket sayısı: 360


In [2]:
ticket_id = "1005127.0"

# Bu ticket'a ait satırları seç.
ticket_satirlari = [
    satir for satir in satirlar
    if satir["issueid"] == ticket_id
]

# Önce yorum, sonra yorum içindeki metin parçasına göre sırala.
ticket_satirlari.sort(
    key=lambda satir: (
        float(satir["comment_seq"]),
        float(satir["utr_seq"])
    )
)

# Her yorumun parçalarını sırayla göster.
onceki_yorum = None

for satir in ticket_satirlari:
    yorum = (satir["comment_seq"], satir["id"])

    if yorum != onceki_yorum:
        print(f"\nYorum {satir['comment_seq']} | {satir['author_role']}")
        onceki_yorum = yorum

    print(satir["actionbody"])


Yorum 0 | reporter
greetings
since couple of weeks we are noticing that  ph_user  url is not opening in normal chrome browser but open in incognito mode.
please check

Yorum 1 | others
.
problem: .
investigation and simulation: .
development agile related issue : .
solution: 1- did we resolve the problem and its symptoms?
2- did we understand and resolve the root cause of the problem?
3- will this problem appear at this customer or other customer site?

Yorum 2 | others
to provide our best support the following policy for tickets waiting customer feedback, and tickets under monitoring will be applied - ph_name team will remind the customer every two working days, after the second reminder if no response, the ph_name team will escalate the ticket and inform the customer that the ticket will be closed by end of second week if no response received.

Yorum 3 | assignee
greetings
please try to clear the browser cache and try to login.
regards,

Yorum 4 | reporter
please close the ticket
af

In [3]:
temiz_ticket = {
    "issueid": ticket_id,
    "problem": (
        "For the past couple of weeks, the URL has not opened "
        "in a normal Chrome window but has opened in incognito mode."
    ),
    "category": "Browser / Access",
    "solution": "Clear the browser cache and try logging in again.",
    "resolution": (
        "The reporter confirmed that it worked after clearing "
        "the browser cache and requested ticket closure."
    ),
    "resolution_confirmed": True,
    "source_comment_seq": {
        "problem": [0],
        "solution": [3],
        "resolution": [4]
    }
}

import json

print(json.dumps(temiz_ticket, indent=2, ensure_ascii=False))

{
  "issueid": "1005127.0",
  "problem": "For the past couple of weeks, the URL has not opened in a normal Chrome window but has opened in incognito mode.",
  "category": "Browser / Access",
  "solution": "Clear the browser cache and try logging in again.",
  "resolution": "The reporter confirmed that it worked after clearing the browser cache and requested ticket closure.",
  "resolution_confirmed": true,
  "source_comment_seq": {
    "problem": [
      0
    ],
    "solution": [
      3
    ],
    "resolution": [
      4
    ]
  }
}


In [4]:
temiz_ticketlar = [temiz_ticket]

def pilotu_kaydet(kayitlar):
    cikti_yolu = Path("tickets_pilot.csv")

    with cikti_yolu.open(
        "w", encoding="utf-8-sig", newline=""
    ) as dosya:
        yazici = csv.DictWriter(
            dosya,
            fieldnames=list(kayitlar[0].keys())
        )
        yazici.writeheader()

        for kayit in kayitlar:
            csv_kaydi = kayit.copy()
            csv_kaydi["source_comment_seq"] = json.dumps(
                kayit["source_comment_seq"]
            )
            yazici.writerow(csv_kaydi)

    print(f"{len(kayitlar)} temiz ticket kaydedildi.")
    print("Dosya:", cikti_yolu.resolve())

pilotu_kaydet(temiz_ticketlar)

1 temiz ticket kaydedildi.
Dosya: D:\ai-muhendis-chatbot\tickets_pilot.csv


In [5]:
import csv
import json
from pathlib import Path

with Path("tickets_pilot_10.csv").open(
    encoding="utf-8-sig", newline=""
) as dosya:
    temiz_ticketlar = list(csv.DictReader(dosya))

for ticket in temiz_ticketlar:
    ticket["source_comment_seq"] = json.loads(
        ticket["source_comment_seq"]
    )
    ticket["resolution_confirmed"] = (
        ticket["resolution_confirmed"].strip().lower() == "true"
    )

print("Pilot kayıt sayısı:", len(temiz_ticketlar))
print(
    "Başarı bildirimi bulunan kayıt:",
    sum(ticket["resolution_confirmed"] for ticket in temiz_ticketlar)
)
print("Alanlar:", list(temiz_ticketlar[0]))

Pilot kayıt sayısı: 10
Başarı bildirimi bulunan kayıt: 8
Alanlar: ['issueid', 'problem', 'category', 'solution', 'resolution', 'resolution_confirmed', 'source_comment_seq', 'solution_scope', 'limitations']
